# 1C–1F — Comparación de encoders en un solo notebook

Ejecuta **de una** los cinco encoders del track inglés (BioBERT, SciBERT,
BiomedBERT-large, BioLinkBERT-large, XLM-RoBERTa-base) con **config idéntica** a 1A/1B
(`lr=2e-5, max_len=256, warmup=300, neg_ratio=3, seed=42, 15 epochs`), guarda
todo en la carpeta de cada uno y al final saca la tabla comparativa con 1A y 1B.

- Los **large** usan `batch=4` (11.5 GB VRAM); los **base**, `batch=16`.
- Entre modelo y modelo se libera la GPU (`empty_cache`).
- **Aviso:** es un run largo (varias horas, sobre todo los large). Déjalo corriendo.
- Cada carpeta queda autocontenida (checkpoint + métricas + artefactos), así que
  si se corta, los modelos ya terminados no se reentrenan: puedes comentar en
  `CONFIGS` los que ya estén hechos y relanzar.

## 1. Setup

In [ ]:
# Ejecucion en servidor local (zape), entorno conda "tfg".
# Override de la cache de HuggingFace a una carpeta escribible del home.
# EJECUTAR ANTES de cualquier import de opennre/transformers/huggingface_hub.
import os
HF_CACHE_DIR = os.path.expanduser("~/hf_cache")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import huggingface_hub.constants as hfc
assert hfc.HF_HUB_CACHE == os.environ["HF_HUB_CACHE"], (
    "Reinicia el kernel y ejecuta esta celda ANTES de cualquier import de HF/opennre.")
print("HF cache:", os.environ["HF_HUB_CACHE"])

# Parches de compatibilidad de OpenNRE (UTF-8, AdamW de torch, num_workers=0)
!python ../baseline/patch_opennre.py

In [ ]:
import json, importlib, time, logging, gc
from collections import Counter
from pathlib import Path
import nltk, pandas as pd, torch, numpy as np

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import opennre

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

## 2. Configuración global y lista de encoders

In [ ]:
# Hiperparametros comunes (identicos a 1A/1B)
MAX_LENGTH   = 256
LEARNING_RATE= 2e-5
EPOCHS       = 15
WARMUP_STEPS = 300
SEED         = 42
NEG_RATIO    = 3

# Datos (formato OpenNRE ya preparado)
DATA_DIR   = Path("../data/english")
TRAIN_DATA = DATA_DIR / "eng_train.txt"
DEV_DATA   = DATA_DIR / "eng_dev.txt"
REL2ID_PATH= DATA_DIR / "rel2id.json"
for p in (TRAIN_DATA, DEV_DATA, REL2ID_PATH):
    assert p.exists(), f"FALTA {p}"
with open(REL2ID_PATH) as f:
    rel2id = json.load(f)
print("Clases:", len(rel2id))

# Los encoders a comparar. Comenta los que ya tengas hechos para no repetir.
# batch=2 en los "large" (bajado de 4): la GPU compartida tiene 10.75 GB reales
# (no los 16 GB de un T4 dedicado), y hubo OOM con batch=4 durante el backward
# de DataParallel. LR/epochs/warmup se mantienen identicos para todos -- el
# batch ya no era "identico" entre base y large de todas formas (16 vs 4).
#
# deberta_v3 y xlnet usan fix_universal_encoder_with_markers() (ver celda
# siguiente), NO fix_universal_encoder() -- este ultimo colapsa los
# marcadores de entidad a [UNK] en DeBERTa-v3 (Bug 2 de
# HALLAZGOS-BUGS-TOKENIZACION.md, comprobado en este repo el 2026-08-03).
# Por eso sus macro_f1 NO son directamente comparables a 1C-1H (biobert,
# scibert, biomedbert_large, biolinkbert_large, xlm_roberta), que se
# entrenaron con el bug sin arreglar -- ver nota en la celda de abajo.
CONFIGS = [
    {"exp": "biobert",           "model": "dmis-lab/biobert-v1.1",                                  "batch": 16, "outdir": "1C-biobert"},
    {"exp": "scibert",           "model": "allenai/scibert_scivocab_uncased",                       "batch": 16, "outdir": "1D-scibert"},
    {"exp": "biomedbert_large",  "model": "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract",  "batch": 2,  "outdir": "1E-biomedbert-large"},
    {"exp": "biolinkbert_large", "model": "michiyasunaga/BioLinkBERT-large",                        "batch": 2,  "outdir": "1F-biolinkbert-large"},
    {"exp": "xlm_roberta",        "model": "xlm-roberta-base",                                       "batch": 16, "outdir": "1H-xlm-roberta"},
    {"exp": "deberta_v3",         "model": "microsoft/deberta-v3-base",                               "batch": 16, "outdir": "1I-deberta-v3"},
    {"exp": "xlnet",              "model": "xlnet-base-cased",                                        "batch": 16, "outdir": "1J-xlnet"},
]
for c in CONFIGS:
    print(f"  {c['exp']:<20} {c['model']:<55} batch={c['batch']}")

## 3. Parche macro_f1 y bucle de entrenamiento

In [ ]:
import sys
sys.path.insert(0, "../baseline")
from patch_opennre import add_macro_f1_metric, fix_universal_encoder_with_markers
add_macro_f1_metric()
# fix_universal_encoder_with_markers() (no fix_universal_encoder()): backbone-
# agnostic COMO fix_universal_encoder, pero ademas usa marcadores de entidad
# dedicados ([E1]/[/E1]/[E2]/[/E2], mean-init) en vez de reutilizar
# [unused0-5] -- fix_universal_encoder() los reutiliza si cls/sep/pad
# "parecen" BERT, pero esa condicion no basta: el tokenizer de DeBERTa-v3 la
# cumple y aun asi sus [unused0-3] colapsan a [UNK] (Bug 2,
# HALLAZGOS-BUGS-TOKENIZACION.md). Tambien incluye el tokenize() consciente
# de anidamiento (Bug 1) y un forward() que no asume pooler_output (necesario
# para DebertaV2Model). Es solo un swap de patch: para biobert/scibert/
# biomedbert_large/biolinkbert_large/xlm_roberta (ya con results_*.json
# cacheado) esta celda no cambia nada porque el loop de mas abajo los salta
# sin reinstanciar BERTEntityEncoder -- solo afecta a deberta_v3 y xlnet, que
# se entrenan de cero.
fix_universal_encoder_with_markers()

from opennre.framework.utils import AverageMeter
from tqdm import tqdm

GRAD_CLIP_NORM = 1.0  # evita la divergencia que vimos en biomedbert_large (loss subiendo, dev colapsando a acc=0)

def train_with_history(fw, max_epoch, metric="macro_f1"):
    """Reimplementa el bucle de SentenceRE para validar y guardar el mejor
    checkpoint por macro_f1 en cada epoch, devolviendo el historial."""
    history, best_metric = [], 0
    for epoch in range(max_epoch):
        fw.train()
        avg_loss, avg_acc = AverageMeter(), AverageMeter()
        t = tqdm(fw.train_loader, desc=f"Epoch {epoch}")
        for data in t:
            if torch.cuda.is_available():
                for i in range(len(data)):
                    try: data[i] = data[i].cuda()
                    except Exception: pass
            label, args = data[0], data[1:]
            logits = fw.parallel_model(*args)
            loss = fw.criterion(logits, label)
            _, pred = logits.max(-1)
            acc = float((pred == label).long().sum()) / label.size(0)
            avg_loss.update(loss.item(), 1); avg_acc.update(acc, 1)
            t.set_postfix(loss=avg_loss.avg, acc=avg_acc.avg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(fw.model.parameters(), GRAD_CLIP_NORM)
            fw.optimizer.step()
            if fw.scheduler is not None: fw.scheduler.step()
            fw.optimizer.zero_grad()
        val = fw.eval_model(fw.val_loader)
        rec = {"epoch": epoch, "train_loss": avg_loss.avg, "train_acc": avg_acc.avg,
               "val_acc": val["acc"], "val_micro_p": val["micro_p"], "val_micro_r": val["micro_r"],
               "val_micro_f1": val["micro_f1"], "val_macro_f1": val["macro_f1"]}
        history.append(rec)
        print(f"Epoch {epoch}: loss={rec['train_loss']:.4f} "
              f"val_micro_f1={rec['val_micro_f1']:.4f} val_macro_f1={rec['val_macro_f1']:.4f}")
        if val[metric] > best_metric:
            print(f"  -> nuevo mejor {metric}={val[metric]:.4f}, guardando checkpoint")
            folder = "/".join(fw.ckpt.split("/")[:-1])
            if folder and not os.path.exists(folder): os.makedirs(folder, exist_ok=True)
            torch.save({"state_dict": fw.model.state_dict()}, fw.ckpt)
            best_metric = val[metric]
    print(f"Mejor {metric} en val: {best_metric:.4f}")
    return history

## 4. Función que ejecuta un encoder completo

In [ ]:
from save_artifacts import dump_all_artifacts

# dev se carga una vez (igual para todos)
dev_instances = [json.loads(l) for l in open(DEV_DATA, encoding="utf-8") if l.strip()]
all_relations = sorted(rel2id.keys())

def _metrics(rows):
    gold = [i["relation"] for i in dev_instances]
    pred = [r["relation"] for r in rows]
    total = len(gold); acc = sum(g==p for g,p in zip(gold,pred))/total
    gc_, pc_, tp_ = Counter(gold), Counter(pred), Counter()
    for g,p in zip(gold,pred):
        if g==p: tp_[g]+=1
    per, f1s = {}, []
    for rel in all_relations:
        tp=tp_.get(rel,0); pt=pc_.get(rel,0); gt=gc_.get(rel,0)
        P=tp/pt if pt else 0; R=tp/gt if gt else 0; F=2*P*R/(P+R) if (P+R) else 0
        per[rel]={"precision":P,"recall":R,"f1":F,"support":gt}
        if gt>0: f1s.append(F)
    return acc, (sum(f1s)/len(f1s) if f1s else 0), per

def run_one_encoder(cfg):
    EXP, MODEL, BATCH = cfg["exp"], cfg["model"], cfg["batch"]
    OUT = Path(f"../outputs/{cfg['outdir']}"); OUT.mkdir(parents=True, exist_ok=True)
    CKPT = OUT / f"eng_{EXP}.pth.tar"; PRED = OUT / f"eng_pred_{EXP}.tsv"
    print("\n" + "#"*70 + f"\n# {EXP}  |  {MODEL}  |  batch={BATCH}\n" + "#"*70)

    encoder = opennre.encoder.BERTEntityEncoder(max_length=MAX_LENGTH, pretrain_path=MODEL)
    model = opennre.model.SoftmaxNN(sentence_encoder=encoder, num_class=len(rel2id), rel2id=rel2id)
    framework = opennre.framework.SentenceRE(
        model=model, train_path=str(TRAIN_DATA), val_path=str(DEV_DATA), test_path=str(DEV_DATA),
        ckpt=str(CKPT), batch_size=BATCH, max_epoch=EPOCHS, lr=LEARNING_RATE,
        opt="adamw", warmup_step=WARMUP_STEPS)
    n_params = sum(p.numel() for p in model.parameters())

    t0 = time.time()
    history = train_with_history(framework, EPOCHS, metric="macro_f1")
    minutes = (time.time()-t0)/60
    with open(OUT / f"history_{EXP}.json", "w") as f: json.dump(history, f, indent=2)

    # recargar el MEJOR checkpoint sobre el mismo modelo (sin duplicar memoria)
    model.load_state_dict(torch.load(str(CKPT), map_location="cpu")["state_dict"])
    if torch.cuda.is_available(): model = model.cuda()
    model.eval()

    # prediccion en dev
    rows = []
    for inst in dev_instances:
        pred_rel, score = model.infer({"text": inst["text"],
            "h": {"pos": inst["h"]["pos"]}, "t": {"pos": inst["t"]["pos"]}})
        rows.append({"document_id": inst["doc_id"], "relation": pred_rel, "score": score,
            "gold": inst["relation"], "head_text": inst["h"]["name"], "head_span": inst["head_span"],
            "head_type": inst["head_type"], "tail_text": inst["t"]["name"],
            "tail_span": inst["tail_span"], "tail_type": inst["tail_type"]})
    pred_df = pd.DataFrame(rows)
    exp_df = pred_df[["document_id","relation","head_text","head_span","head_type",
                      "tail_text","tail_span","tail_type"]]
    exp_df[exp_df["relation"]!="no_relation"].to_csv(PRED, sep="\t", index=False)

    acc, macro_f1, per = _metrics(rows)
    best = max(history, key=lambda h: h["val_macro_f1"])
    results = {"experiment": EXP, "model": MODEL,
        "hyperparameters": {"max_length":MAX_LENGTH,"batch_size":BATCH,"learning_rate":LEARNING_RATE,
            "epochs":EPOCHS,"warmup_steps":WARMUP_STEPS,"neg_ratio":NEG_RATIO,"seed":SEED},
        "results": {"accuracy":acc, "macro_f1":macro_f1, "micro_f1":best["val_micro_f1"],
            "best_epoch":best["epoch"], "per_relation":per},
        "n_params": int(n_params), "train_minutes": round(minutes,1)}
    with open(OUT / f"results_{EXP}.json", "w") as f: json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"[{EXP}] Macro F1 dev = {macro_f1:.4f} | micro = {best['val_micro_f1']:.4f} | {minutes:.1f} min")

    # artefactos extra (run_meta, dev_probs/gold, preds detalladas, confusion.csv, bootstrap CI)
    hparams = results["hyperparameters"]
    try:
        dump_all_artifacts(model, encoder, dev_instances, pred_df, rel2id,
                           OUT, EXP, MODEL, hparams, history=history, batch_size=BATCH)
    except Exception as e:
        import traceback; traceback.print_exc(); print("AVISO artefactos:", e)

    summary = {"exp":EXP, "model":MODEL, "macro_f1":macro_f1, "micro_f1":best["val_micro_f1"],
               "accuracy":acc, "best_epoch":best["epoch"], "n_params":int(n_params), "minutes":round(minutes,1)}
    # liberar GPU antes del siguiente modelo
    del framework, model, encoder; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return summary

## 5. Ejecutar los 4 encoders

In [ ]:
MACRO_F1_SANITY_FLOOR = 0.5  # muy por debajo de un run sano (~0.75-0.80), muy por encima de un run roto (~0.02)

summaries = []
for cfg in CONFIGS:
    OUT = Path(f"../outputs/{cfg['outdir']}")
    done = (OUT / f"results_{cfg['exp']}.json")
    if done.exists():
        prev = json.load(open(done))["results"]
        if prev["macro_f1"] < MACRO_F1_SANITY_FLOOR:
            print(f"[AVISO] {cfg['exp']}: results_*.json existe pero macro_f1={prev['macro_f1']:.4f} "
                  f"esta por debajo del umbral de cordura ({MACRO_F1_SANITY_FLOOR}) -- huele a run roto "
                  f"(mismo patron que rompio 1B-biolinkbert: loss/accuracy planos). Reentrenando en vez de hacer skip.")
            summaries.append(run_one_encoder(cfg))
            continue
        print(f"[skip] {cfg['exp']} ya tiene results_*.json (macro_f1={prev['macro_f1']:.4f}) -> lo cargo sin reentrenar")
        summaries.append({**prev, "exp":cfg["exp"], "model":cfg["model"]})
        continue
    summaries.append(run_one_encoder(cfg))

print("\nTODOS LOS ENCODERS COMPLETADOS")

## 5b. Validación en modo blind (protocolo oficial CodaBench)

Igual que en 1A/1B: `dev_instances` (usado arriba) es el dev curado, sin
`no_relation`. El protocolo real enumera todos los pares candidatos de
`eng_dev_blind.txt` (282k) y puntúa con `baseline/score.py` contra el gold real
(`eng-dev-rel.tsv`). Reutiliza los checkpoints ya entrenados -- no reentrena
nada. Resumible igual que el entrenamiento: si ya existe `results_blind_*.json`
lo salta.

In [ ]:
from score import evaluate

BLIND_DEV_PATH = DATA_DIR / "eng_dev_blind.txt"
BLIND_GOLD_TSV = DATA_DIR / "eng-dev-rel.tsv"
for p in (BLIND_DEV_PATH, BLIND_GOLD_TSV):
    assert p.exists(), f"FALTA {p}"

blind_raw = [json.loads(l) for l in open(BLIND_DEV_PATH, encoding="utf-8") if l.strip()]
gold_df_blind = pd.read_csv(BLIND_GOLD_TSV, sep="\t")
print(f"candidatos blind: {len(blind_raw)} | gold real: {len(gold_df_blind)}")

def batched_predict_labels(model, encoder, instances, batch_size, device):
    """Igual que la inferencia en dev de mas arriba, pero en lotes -- 282k
    candidatos uno a uno con model.infer() tardaria demasiado."""
    id2rel_local = {v: k for k, v in rel2id.items()}
    preds = []
    with torch.no_grad():
        for s in range(0, len(instances), batch_size):
            batch = instances[s:s + batch_size]
            tok = [encoder.tokenize({"text": i["text"], "h": {"pos": i["h"]["pos"]},
                                     "t": {"pos": i["t"]["pos"]}}) for i in batch]
            fields = [torch.cat([t[k] for t in tok], dim=0).to(device) for k in range(len(tok[0]))]
            logits = model(*fields)
            preds.extend(id2rel_local[i] for i in logits.argmax(-1).cpu().numpy())
    return preds

def run_blind_eval(cfg, batch_size=32):
    EXP, MODEL = cfg["exp"], cfg["model"]
    OUT = Path(f"../outputs/{cfg['outdir']}")
    CKPT = OUT / f"eng_{EXP}.pth.tar"
    BLIND_RESULTS = OUT / f"results_blind_{EXP}.json"
    BLIND_PRED_TSV = OUT / f"eng_pred_blind_{EXP}.tsv"

    if BLIND_RESULTS.exists():
        print(f"[skip-blind] {EXP} ya tiene results_blind_*.json")
        return json.load(open(BLIND_RESULTS))
    if not CKPT.exists():
        print(f"[AVISO] {EXP}: no hay checkpoint todavia (entrenamiento no terminado), se omite blind")
        return None

    print(f"\n[blind] {EXP} -- prediciendo sobre {len(blind_raw)} candidatos...")
    encoder = opennre.encoder.BERTEntityEncoder(max_length=MAX_LENGTH, pretrain_path=MODEL)
    model = opennre.model.SoftmaxNN(sentence_encoder=encoder, num_class=len(rel2id), rel2id=rel2id)
    model.load_state_dict(torch.load(str(CKPT), map_location="cpu")["state_dict"])
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device); model.eval()

    t0 = time.time()
    pred_labels = batched_predict_labels(model, encoder, blind_raw, batch_size, device)
    minutes = (time.time() - t0) / 60

    rows = [
        {"document_id": inst["doc_id"], "relation": rel,
         "head_text": inst["h"]["name"], "head_span": inst["head_span"], "head_type": inst["head_type"],
         "tail_text": inst["t"]["name"], "tail_span": inst["tail_span"], "tail_type": inst["tail_type"]}
        for inst, rel in zip(blind_raw, pred_labels) if rel != "no_relation"
    ]
    pd.DataFrame(rows).to_csv(BLIND_PRED_TSV, sep="\t", index=False)

    res = evaluate(pd.read_csv(BLIND_PRED_TSV, sep="\t"), gold_df_blind)
    out = {"experiment": EXP, "model": MODEL, "macro_f1_ciego": res["macro_f1"],
           "micro_f1_ciego": res["micro_f1"], "accuracy_ciego": res["accuracy"],
           "n_predicted": len(rows), "n_candidates": len(blind_raw),
           "inference_minutes": round(minutes, 1)}
    with open(BLIND_RESULTS, "w") as f:
        json.dump(out, f, indent=2, ensure_ascii=False)
    print(f"[blind] {EXP}: macro_f1_ciego={res['macro_f1']:.4f}  ({minutes:.1f} min, "
          f"{len(rows)} predicciones de relacion)")

    del model, encoder; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return out

blind_summaries = [r for r in (run_blind_eval(cfg) for cfg in CONFIGS) if r]
print("\nVALIDACION BLIND COMPLETADA (para los que ya tenian checkpoint)")

## 6. Tabla comparativa final (con 1A y 1B)

In [ ]:
# Reune 1A, 1B y los 4 nuevos desde sus results_*.json, con curado y ciego
def load_result(path, blind_path, exp):
    p = Path(path)
    if not p.exists(): return None
    d = json.load(open(p))["results"]
    row = {"exp": exp, "macro_f1": d["macro_f1"], "micro_f1": d.get("micro_f1"),
           "best_epoch": d.get("best_epoch"), "macro_f1_ciego": None}
    bp = Path(blind_path)
    if bp.exists():
        row["macro_f1_ciego"] = json.load(open(bp))["macro_f1_ciego"]
    return row

rows = [
    load_result("../outputs/1A-pubmedbert/results_pubmedbert.json",
                "../outputs/1A-pubmedbert/results_blind_pubmedbert.json", "1A PubMedBERT"),
    load_result("../outputs/1B-biolinkbert/results_biolinkbert_base.json",
                "../outputs/1B-biolinkbert/results_blind_biolinkbert_base.json", "1B BioLinkBERT-base"),
]
for cfg in CONFIGS:
    rows.append(load_result(f"../outputs/{cfg['outdir']}/results_{cfg['exp']}.json",
                            f"../outputs/{cfg['outdir']}/results_blind_{cfg['exp']}.json",
                            f"{cfg['outdir']}"))
rows = [r for r in rows if r]
tab = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
tab["diff_vs_baseline"] = tab["macro_f1"] - 0.6944
print(tab.to_string(index=False))

Path("../outputs").mkdir(exist_ok=True)
tab.to_csv("../outputs/comparacion_encoders.csv", index=False)
print("\nGuardado: ../outputs/comparacion_encoders.csv")

## 7. Multi-seed pareado -- ¿el ranking curado-vs-ciego es real o ruido de seed?

**Motivo:** `run_one_encoder` nunca fija seed de verdad -- `"seed": 42` en los
`results_*.json` es solo una etiqueta, no siembra `random`/`numpy`/`torch`. Por
eso el mismo config de BioLinkBERT-base dio 0.7835/0.308 en un run y
0.8172/0.273 en otro: sin seed real, cada entrenamiento es una muestra distinta.

**Diseño pareado:** las mismas 3 seeds (42, 123, 2024) para los 4 encoders base
(PubMedBERT, BioLinkBERT-base, BioBERT, SciBERT) -- no seeds sueltas por
modelo. Cada (seed, encoder): entrena con seed fijada de verdad, evalúa en
ciego, barre threshold hasta 0.99 (el grid recortado a 0.95 ya nos dio un
óptimo falso una vez). Resumible: si ya existe `results_seed_summary.json` para
esa combinación, la salta.

**Coste:** 3 seeds x 4 encoders = 12 entrenamientos completos (~30-40 min cada
uno en batch=16) + inferencia blind (~20 min) + barrido de threshold (rápido,
ya cacheado) = varias horas. Pensado para dejarlo corriendo largo rato.

In [ ]:
import random

def set_seed(seed):
    """Siembra de verdad -- lo que faltaba en todo el pipeline hasta ahora."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEEDS = [42, 123, 2024]

MULTISEED_CONFIGS = [
    {"exp": "pubmedbert",       "model": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext", "batch": 16, "base_outdir": "1A-pubmedbert"},
    {"exp": "biolinkbert_base", "model": "michiyasunaga/BioLinkBERT-base",                                 "batch": 16, "base_outdir": "1B-biolinkbert"},
    {"exp": "biobert",          "model": "dmis-lab/biobert-v1.1",                                           "batch": 16, "base_outdir": "1C-biobert"},
    {"exp": "scibert",          "model": "allenai/scibert_scivocab_uncased",                                "batch": 16, "base_outdir": "1D-scibert"},
]

NO_REL_ID = rel2id["no_relation"]
THRESH_GRID = [round(float(x), 2) for x in np.arange(0.05, 0.951, 0.05)] + [0.96, 0.97, 0.98, 0.99]

def _preds_at_threshold(probs, threshold):
    probs2 = probs.copy()
    probs2[:, NO_REL_ID] = -1
    best_rel_id = probs2.argmax(axis=1)
    best_rel_prob = probs[np.arange(len(probs)), best_rel_id]
    return np.where(best_rel_prob >= threshold, best_rel_id, NO_REL_ID)

def _rows_from_preds(pred_ids, id2rel_local):
    labels = [id2rel_local[i] for i in pred_ids]
    return [
        {"document_id": inst["doc_id"], "relation": rel,
         "head_text": inst["h"]["name"], "head_span": inst["head_span"], "head_type": inst["head_type"],
         "tail_text": inst["t"]["name"], "tail_span": inst["tail_span"], "tail_type": inst["tail_type"]}
        for inst, rel in zip(blind_raw, labels) if rel != "no_relation"
    ]

def run_one_seed_full(cfg, seed):
    EXP, MODEL, BATCH = cfg["exp"], cfg["model"], cfg["batch"]
    OUT = Path(f"../outputs/{cfg['base_outdir']}/seed{seed}")
    OUT.mkdir(parents=True, exist_ok=True)
    CKPT = OUT / f"eng_{EXP}.pth.tar"
    SUMMARY_PATH = OUT / "results_seed_summary.json"

    if SUMMARY_PATH.exists():
        print(f"[skip] {EXP} seed={seed} ya tiene results_seed_summary.json")
        return json.load(open(SUMMARY_PATH))

    print("\n" + "#" * 70 + f"\n# {EXP}  seed={seed}\n" + "#" * 70)
    set_seed(seed)

    encoder = opennre.encoder.BERTEntityEncoder(max_length=MAX_LENGTH, pretrain_path=MODEL)
    model = opennre.model.SoftmaxNN(sentence_encoder=encoder, num_class=len(rel2id), rel2id=rel2id)
    framework = opennre.framework.SentenceRE(
        model=model, train_path=str(TRAIN_DATA), val_path=str(DEV_DATA), test_path=str(DEV_DATA),
        ckpt=str(CKPT), batch_size=BATCH, max_epoch=EPOCHS, lr=LEARNING_RATE,
        opt="adamw", warmup_step=WARMUP_STEPS)

    t0 = time.time()
    history = train_with_history(framework, EPOCHS, metric="macro_f1")
    train_minutes = (time.time() - t0) / 60
    with open(OUT / f"history_{EXP}.json", "w") as f:
        json.dump(history, f, indent=2)
    best = max(history, key=lambda h: h["val_macro_f1"])
    macro_f1_curado = best["val_macro_f1"]

    # recargar el mejor checkpoint para blind + calibracion
    model.load_state_dict(torch.load(str(CKPT), map_location="cpu")["state_dict"])
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device); model.eval()
    id2rel_local = {v: k for k, v in rel2id.items()}

    t0 = time.time()
    all_probs = np.zeros((len(blind_raw), len(rel2id)), dtype=np.float32)
    with torch.no_grad():
        for s in range(0, len(blind_raw), 64):
            batch = blind_raw[s:s + 64]
            tok = [encoder.tokenize({"text": i["text"], "h": {"pos": i["h"]["pos"]},
                                     "t": {"pos": i["t"]["pos"]}}) for i in batch]
            fields = [torch.cat([t[k] for t in tok], dim=0).to(device) for k in range(len(tok[0]))]
            logits = model(*fields)
            all_probs[s:s + len(batch)] = torch.softmax(logits, dim=-1).cpu().numpy()
    blind_minutes = (time.time() - t0) / 60
    np.save(OUT / f"blind_probs_{EXP}.npy", all_probs)

    argmax_ids = all_probs.argmax(axis=1)
    res_argmax = evaluate(pd.DataFrame(_rows_from_preds(argmax_ids, id2rel_local)), gold_df_blind)

    sweep_rows = []
    for th in THRESH_GRID:
        pred_ids = _preds_at_threshold(all_probs, th)
        res = evaluate(pd.DataFrame(_rows_from_preds(pred_ids, id2rel_local)), gold_df_blind)
        sweep_rows.append({"threshold": th, "macro_f1": res["macro_f1"]})
    sweep_df = pd.DataFrame(sweep_rows)
    best_row = sweep_df.loc[sweep_df["macro_f1"].idxmax()]
    best_threshold = float(best_row["threshold"])
    at_edge = best_threshold == THRESH_GRID[-1]

    summary = {
        "exp": EXP, "model": MODEL, "seed": seed,
        "macro_f1_curado": macro_f1_curado,
        "macro_f1_ciego_argmax": res_argmax["macro_f1"],
        "best_threshold": best_threshold,
        "macro_f1_ciego_calibrado": float(best_row["macro_f1"]),
        "threshold_en_borde_del_grid": at_edge,
        "train_minutes": round(train_minutes, 1),
        "blind_minutes": round(blind_minutes, 1),
    }
    with open(SUMMARY_PATH, "w") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"[{EXP} seed={seed}] curado={macro_f1_curado:.4f}  ciego_argmax={res_argmax['macro_f1']:.4f}  "
          f"ciego_calibrado={summary['macro_f1_ciego_calibrado']:.4f} (th={best_threshold:.2f}"
          f"{', BORDE DEL GRID -- revisar' if at_edge else ''})")

    del framework, model, encoder; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return summary

multiseed_summaries = []
for cfg in MULTISEED_CONFIGS:
    for seed in SEEDS:
        multiseed_summaries.append(run_one_seed_full(cfg, seed))

print("\nMULTI-SEED COMPLETADO")

In [ ]:
# ============================================================
# AGREGACION: media +/- std por encoder, ranking seed a seed,
# y diferencia pareada (por seed) entre cada par de encoders
# ============================================================
all_seed_rows = []
for cfg in MULTISEED_CONFIGS:
    for seed in SEEDS:
        p = Path(f"../outputs/{cfg['base_outdir']}/seed{seed}/results_seed_summary.json")
        if p.exists():
            all_seed_rows.append(json.load(open(p)))

df = pd.DataFrame(all_seed_rows)
if df.empty:
    print("Todavia no hay resultados de multi-seed guardados.")
else:
    if df["threshold_en_borde_del_grid"].any():
        print("AVISO: algun best_threshold cayo en el borde del grid (0.99) -- "
              "el verdadero optimo podria estar mas alla, igual que paso antes con 0.95.\n")

    print("=== Media +/- std por encoder (ciego calibrado) ===")
    agg = df.groupby("exp")["macro_f1_ciego_calibrado"].agg(["mean", "std", "count"])
    agg = agg.sort_values("mean", ascending=False)
    print(agg.to_string())

    print("\n=== Ranking seed a seed (ciego calibrado) ===")
    pivot = df.pivot(index="seed", columns="exp", values="macro_f1_ciego_calibrado")
    print(pivot.to_string())
    rankings = pivot.rank(axis=1, ascending=False)
    ranking_stable = (rankings.nunique() == 1).all()
    print(f"\n¿Ranking identico en las {len(SEEDS)} seeds? {'SI' if ranking_stable else 'NO'}")

    print("\n=== Diferencias pareadas por seed (col - fila) ===")
    exps = pivot.columns.tolist()
    for i, a in enumerate(exps):
        for b in exps[i+1:]:
            diffs = pivot[b] - pivot[a]
            signo = "SIEMPRE +" if (diffs > 0).all() else ("SIEMPRE -" if (diffs < 0).all() else "CAMBIA DE SIGNO")
            print(f"{b} - {a}: {diffs.values.round(4).tolist()}  -> {signo}")

    df.to_csv("../outputs/multiseed_resultados.csv", index=False)
    print("\nGuardado: ../outputs/multiseed_resultados.csv")